<a href="https://colab.research.google.com/github/mikakia/BreastCancer/blob/main/Breast_Cancer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import  missingno as msno

import seaborn as sns

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC, NuSVC
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.decomposition import PCA

from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import KFold, cross_val_score,cross_val_predict

In [ ]:
!pip3 install -U ucimlrepo

In [ ]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
breast_cancer_wisconsin_diagnostic = fetch_ucirepo(id=17)

# data (as pandas dataframes)
X = breast_cancer_wisconsin_diagnostic.data.features
y = breast_cancer_wisconsin_diagnostic.data.targets

# metadata
print(breast_cancer_wisconsin_diagnostic.metadata)

# variable information
#print(breast_cancer_wisconsin_diagnostic.variables)


In [ ]:
feature_names = X.columns
feature_names = ['id', 'diagnosis'] + feature_names.tolist()
print(feature_names)

#Exploring Dataset

In [ ]:
url = "https://raw.githubusercontent.com/mikakia/BreastCancer/main/wdbc.data"
df = pd.read_csv(url, sep=",",header=None,names=feature_names)
df.head()

In [ ]:
df.dtypes


In [ ]:
df.info()

In [ ]:
df.isna().sum()

#Preprocessing

In [ ]:
df.describe()

###Plots

In [ ]:
df_no_id_dg = df.drop(['id', 'diagnosis'], axis=1)

plt.figure(figsize=(15,6))
sns.boxplot(data=df_no_id_dg)
plt.xticks(rotation=90)
plt.title("Boxplot of WDBC features")
plt.show()


In [ ]:
sns.scatterplot(x='radius1', y='area1', data=df)
plt.title("Radius vs Area")
plt.show()

##Outlier check

In [ ]:
print("Min value:", df['area1'].min())
print("Max value:", df['area1'].max())

In [ ]:
#check for outliers in areas (mean per patient)
area_columns = ['area1', 'area2', 'area3']
mean_area_per_patient = df[area_columns].mean(axis=1)  # axis=1 means row-wise
print(mean_area_per_patient.max())

In [ ]:
#check for outliers in radius (mean per patient)
area_columns = ['radius1', 'radius2', 'radius3']
mean_area_per_patient = df[area_columns].mean(axis=1)  # axis=1 means row-wise
print(mean_area_per_patient.min())


The mean of smoothness is 0.12 > 0.2 where 0.2 the max expected mean. It can be measurement error.

In [ ]:
#check for outliers in smoothness (mean per patient)
area_columns = ['smoothness1', 'smoothness2', 'smoothness3']
mean_area_per_patient = df[area_columns].mean(axis=1)  # axis=1 means row-wise
print(mean_area_per_patient.max())


###Check the outliers ranges for smoothness1,2,3
normal extreme values indicating suspicious/malignant cell

In [ ]:
cols = ['smoothness1','smoothness2','smoothness3']

for col in cols:

  Q1 = df[col].quantile(0.25)
  Q3 = df[col].quantile(0.75)
  IQR = Q3 - Q1
  lower_bound = Q1 - 1.5 * IQR
  upper_bound = Q3 + 1.5 * IQR

  # Show outlier rows
  outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
  print(outliers[[col]])


##Standarlization

In [ ]:


features = df.drop(['id', 'diagnosis'], axis=1)
scaler = StandardScaler()

scaled_features = scaler.fit_transform(features)
scaled_df_sel_feat= pd.DataFrame(scaled_features, columns=features.columns)

df_scaled = pd.concat([df[['id', 'diagnosis']], scaled_df_sel_feat], axis=1)

df_scaled.head()



#Training and Test

In [ ]:
X = df_scaled.drop(['id', 'diagnosis'], axis=1)
y = df_scaled['diagnosis']

In [ ]:
X_notscaled = df.drop(['id', 'diagnosis'], axis=1)
y_notscaled = df['diagnosis']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
X_train_ns, X_test_ns, y_train_ns, y_test_ns = train_test_split(X_notscaled, y_notscaled, test_size=0.2, random_state=42, stratify=y_notscaled)

##KNN with k folds
Accuracy: 97%

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
kf = KFold(n_splits=14, shuffle=True, random_state=42)

In [ ]:
scores = cross_val_score(knn, X, y, cv=kf, scoring='accuracy')

print("Accuracy for each fold:", scores)
print("Mean accuracy:", scores.mean())

In [ ]:
y_pred = cross_val_predict(knn, X, y, cv=kf)
results = pd.DataFrame({'True': y, 'Predicted': y_pred})
print(results.head(20))

###KNN without kfolds
Accuracy: 96%

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train, y_train)

In [ ]:
y_pred_knn = knn_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_knn))
print("Classification Report:\n", classification_report(y_test, y_pred_knn))
print("Confusion Matric\n",confusion_matrix(y_test, y_pred_knn))

##Random Forest
Accuracy: 97%

In [ ]:
rf_model = RandomForestClassifier(n_estimators=30,max_depth=None,random_state=42)
rf_model.fit(X_train_ns, y_train_ns)

In [ ]:
y_pred_rf = rf_model.predict(X_test_ns)
print("Accuracy:", accuracy_score(y_test_ns, y_pred_rf))
print("\nClassification Report:\n", classification_report(y_test_ns, y_pred_rf))

##Neural Network
Accuracy: 97%%

In [ ]:

nn_model = MLPClassifier(hidden_layer_sizes=(30,15), activation='relu', max_iter=500, random_state=42)
nn_model.fit(X_train, y_train)

In [ ]:
y_pred_nn = nn_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_nn))
print("Classification Report:\n", classification_report(y_test, y_pred_nn))

##SVM with rbf
Accuracy: 97%

In [ ]:
svm_model = SVC(kernel='rbf', C=1.0, probability=True, random_state=42)
svm_model.fit(X_train, y_train)

In [ ]:
y_pred = svm_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred_nn))
print("Classification Report:")
print(classification_report(y_test, y_pred))